In [ ]:
# return과 yield의 차이 (함수가 끝나는지, 멈추는지)

def with_return():
    print("A")
    return 1
    print("B")   # 실행되지 않음

# 파이썬 인터프리터는 함수 정의를 만나면 함수 객체를 생성; yield가 있든 없든 동일함
# 함수 객체가 있어야 함수 호출이 가능함, 차이는 함수를 호출하는 시점에 발생
# 일반 함수는 호출하면 결과값을 반환한 다음 종료
# yield가 포함된 함수를 호출하면, 함수 본문이 즉시 실행되는 것이 아니라 제너레이터 객체를 만들어서 반환
# 반환된 제너레이터 객체의 next()를 호출하면 첫 yield까지 실행되고 멈춤
def with_yield():
    print("A")
    yield 1
    print("B")

# return 함수 호출 - 즉시 끝까지 실행되고 반환
result = with_return()
print("result:", result)

print("-"*100)

# yield가 포함된 함수 호출 → 함수 본문이 실행되지 않고, 제너레이터 객체가 생성됨
gen = with_yield()
print("gen:", gen)

A
result: 1
----------------------------------------------------------------------------------------------------
gen: <generator object with_yield at 0x0000026009794B80>


In [ ]:
# yield가 들어있는 함수를 호출하면 함수가 실행되는 것이 아니라 제너레이터 객체가 생성됨

# yield가 포함된 함수 실행은 어떻게? next(generator)로 실행
# next(generator)는 generator에게 "다음 값을 달라"고 요청을 하는 것
# 다음 값을 달라고 하면 제너레이터는 yield ■를 실행해서 ■를 반환
# return은 값을 내보내고 함수를 완전히 종료하는데 반해
# yield ■는 값 ■을 내보내고 그 자리에서 멈춤 → 해당 함수는 끝난 것이 아니며
# next()가 호출되면 멈춘 지점부터 다시 이어서 실행

# ----- yield A ----- yield B ----- yield C ----- yield D -----
# ------------- 
# next() → A
#               ------------- 
#               next() → B
#                             ------------- 
#                             next() → C
#                                           ------------- 
#                                           next() → D
#                                                         ----- 
#                                                         next() → yield D 이후의 코드를 실행
#                                                         함수의 끝에 도달, StopIteration이라는 예외를 자동으로 발생

# next()를 호출하면 다음 yield 까지 사이에 있는 코드를 전부 실행
# yield 실행 후 멈춤
# yield ★의 ★는 next()의 리턴으로 반환

# 따라서 next()를 yield 개수만큼 호출하면 값을 다 받을 수 있고
# 그 보다 한 번 더 호출하면 StopIteration이라는 예외가 발생
# StopIteration은 제너레이터가 소진되었음을 알리는 정상적인 종료 신호

# def gen_func():
#     print("A")
#     yield 1
#     print("B")
#     yield 2
#     print("C")

# gen = gen_func()  # yield가 포함된 함수이기 때문에 제네레이터 객체를 생성해서 gen으로 리턴
# next(gen)         # "A" 출력, yield 1에서 멈춤, 1 반환
# next(gen)         # "B" 출력, yield 2에서 멈춤, 2 반환
# next(gen)         # "C" 출력, 더 이상 yield가 없어서 함수 끝까지 실행 → StopIteration 발생

# StopIteration 예외를 통해 함수 내의 모든 yield 실행이 종료되었음을 알 수 있는 것

In [ ]:
# for 문으로 제너레이터를 순회

def gen_func():
    print("A")
    yield 1
    print("B")
    yield 2
    print("C")
    yield 3

gen2 = gen_func()
for value in gen2:      
    # value = next(gen2) 반복을 자동으로 실행
    # for 문은 StopIteration 예외가 발생하면 종료
    print("받은 값:", value)

A
받은 값: 1
B
받은 값: 2
C
받은 값: 3


In [ ]:
# A    --Depends(B)--> B
#      <----------     B
# 문제  ----------->   B

In [ ]:
# raise는 호출 스택을 거슬러 올라가면서 처리할 대상을 찾음

def a():
    raise ValueError("에러")    # 4. raise

def b():
    a()                         # 3. a를 호출

def c():
    b()                         # 2. b를 호출

try:
    c()                         # 1. c를 호출
except ValueError as e:
    print("에러:", e)

# 호출 스택
# c() 호출 → c() 안에서 b() 호출 → b() 안에서 a() 호출 → a() 안에서 raise

# a()에서 raise 발생
#     ↓ 
# a 내부에는 except 없음 → a를 호출한   b로 전달
# b 내부에는 except 없음 → b를 호출한   c로 전달
# c 내부에는 except 없음 → c를 호출한 try로 전달

# try/except에서 예외 처리 → "에러: e" 출력

에러: 에러


In [ ]:
# 내부에서 예외를 처리하고 caller에게 그 소식을 알림

def f():
    try:
        # ................
        raise ValueError("에러")
    except ValueError as e:
        print("f 내부에서 받음:", e)
        raise
        # 동일한 예외를 다시 발생시켜서 호출 스택의 위쪽으로 전파
        # 인자 없는 raise는 반드시 try, except 블록 내에서만 사용해야 함
        # 그렇게 해야 현재 처리 중인 예외를 바깥으로 전달할 수 있음

# ---------------------------------------------------------------

try:
    f()
except ValueError as e:
    print("바깥에서 처리:", e)

# ⓐ f 내부에서 실행 중 예외가 발생
# ⓑ f 내부에서 해당 예외를 처리
# ⓒ raise를 통해 해당 예외를 바깥으로 전달
# ⓓ 해당 예외를 처리할 수 있는 try, except가 "바깥"에 있어야 하며 → 이 경우 try: 부터 실행되는 흐름
#    만약 없다면 누군가 try/except로 처리할 때까지 계속 위로(호출한 쪽으로) 전파

f 내부에서 받음: 에러
바깥에서 처리: 에러


In [ ]:

def gen_func():
    print("A")
    try:
        yield 1 # ←-------------------------------------------- 여기에서 예외가 발생할 이유가 없음
                #                                               하지만 예외를 발생시키고 싶다면? ValueError("throw로 전달한 예외")↑
                #                                               예외가 발생되면 except 블록이 실행
    except ValueError as e:
        print("제너레이터 내부에서 예외를 처리:", e)
        # raise   ←-------------------------------------------- 발생 예외가 다시 외부로 전파 ★★★★★
    print("B")
    yield 2

# -----------------------------------------------------------------------------------

try:
    raise ValueError("일반적인 raise")
except ValueError as e:
    print("메인 흐름에서 예외를 처리:", e)



메인 흐름에서 예외를 처리: 일반적인 raise


In [ ]:
# 제네레이터.throw(): 멈춰있는 제너레이터의 yield 지점에 예외를 주입함
# "멈춰있는 제너레이터의 yield 지점에서 예외가 발생한 것처럼 하라"

gen = gen_func()

print(next(gen))                                    # "A" 출력 -> yield 1에서 멈춤 -> 1 반환
print('='*100)                                      
# 받아간 1을 가지고 뭔가 작업을 함, 이 지점에서 뭔가 문제가 생김
print(gen.throw(ValueError("throw로 전달한 예외")))   # 그렇다면 제네레이터에게 알려서 그에 맞는 뒷 처리를 하게 해야 함


# 1. next()를 한 번이라도 호출해서 제너레이터가 어떤 yield 지점에서 멈춰있는 상태에서
#    generator.throw(■)를 호출하면 멈춰있던 yield 위치에서 ■ 예외가 발생한 것처럼 처리

# 왜 그렇게 함 ???
# yield ■가 실행되어서 ■로 어떤 작업을 하다가 문제가 생김
# 문제가 생긴 상황에서 yield 이후의 실행은 문제 상황에 맞게 처리를 하고 싶기 때문임
# 따라서 외부에서 yield 지점으로 예외를 "날리는" 것

# 2. 제너레이터 내부에 except가 없는 경우 예외 ■는 throw()를 호출한 지점으로 그대로 돌아옴
#    raise가 없어도, 예외는 호출 스택을 거슬러 올라가기 때문
#    raise가 필요한 경우는, except로 예외 처리를 했음에도, 다시 바깥으로 던지고 싶을 때

#    제네레이터에는 문제 상황에 대비한 try except 가 준비되어 있어야 함
#    그렇지 않으면 throw()를 호출한 지점으로 되돌가 가기 때문

# 3. 제너레이터 내부에서 except로 처리한 뒤 raise를 사용하지 않으면 해당 예외는 소멸
# 4. raise를 실행해서 바깥에 알려야 할 필요가 있는 경우 해당 예외가 다시 throw 지점으로 전달
# →
# "문제 상황이 생겼으니까 일단 제네레이터 너가 내부에서 해결하고 나서
# 나도 해결해야 하니까, 너 예외 처리 끝나면 나한테도 다시 알려줘, 그러면 나의 try except가 실행될거야"


A
1
제너레이터 내부에서 예외를 처리: throw로 전달한 예외
B
2
